# Phase 4A — deterministic training calibration

This notebook is a thin Colab orchestrator. It reads targets only from the frozen `train` split, verifies that both feature profiles are aligned, runs the full test suite, and writes the reproducible `calibration_manifest.json` to Drive. It does not train a model or access validation/test labels.

In [1]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
        check=True,
    )

sys.path.insert(0, str(REPO_ROOT / 'code/python'))

In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
GRAPH_VERSION = 'infiltration_v1_w30_tcpflags_episode_split_v1'
GRAPH_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain/graphs') / GRAPH_VERSION
CALIBRATION_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain/calibration') / GRAPH_VERSION
CALIBRATION_MANIFEST = CALIBRATION_ROOT / 'calibration_manifest.json'

# Set only if the corrected-data manifest was moved after graph construction.
CORRECTED_MANIFEST_OVERRIDE = None
VERIFY_GRAPH_CHECKSUMS = False  # Enable after copying/moving the graph collection.
OVERWRITE_DIFFERENT_MANIFEST = False  # Identical reruns never require overwrite.

assert (GRAPH_ROOT / 'graph_manifest.json').is_file(), GRAPH_ROOT
print({'graph_root': str(GRAPH_ROOT), 'output': str(CALIBRATION_MANIFEST)})

{'graph_root': '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_episode_split_v1', 'output': '/content/drive/MyDrive/nids-fair-retrain/calibration/infiltration_v1_w30_tcpflags_episode_split_v1/calibration_manifest.json'}


## Acceptance tests

Run the complete suite against the exact checked-out revision before publishing the calibration artifact.

In [4]:
tests_root = REPO_ROOT / 'code/python/tests'
test_env = dict(__import__('os').environ)
test_env['PYTHONPATH'] = str(REPO_ROOT / 'code/python')
subprocess.run(
    [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', str(tests_root), '-p', 'test_*.py', '-v',
    ],
    cwd=REPO_ROOT,
    env=test_env,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', '/content/temporalgnn-nids/code/python/tests', '-p', 'test_*.py', '-v'], returncode=0)

## Generate the frozen candidate grid

The CLI refuses to replace a different existing artifact unless the overwrite flag is deliberately enabled. An identical rerun reports `unchanged`.

In [5]:
command = [
    sys.executable,
    str(REPO_ROOT / 'code/python/scripts/calibrate_pos_weight.py'),
    '--graph-root', str(GRAPH_ROOT),
    '--output', str(CALIBRATION_MANIFEST),
]
if CORRECTED_MANIFEST_OVERRIDE is not None:
    command.extend(['--corrected-manifest', str(CORRECTED_MANIFEST_OVERRIDE)])
if VERIFY_GRAPH_CHECKSUMS:
    command.append('--verify-checksums')
if OVERWRITE_DIFFERENT_MANIFEST:
    command.append('--overwrite')

subprocess.run(command, cwd=REPO_ROOT, check=True)

CompletedProcess(args=['/usr/bin/python3', '/content/temporalgnn-nids/code/python/scripts/calibrate_pos_weight.py', '--graph-root', '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_tcpflags_episode_split_v1', '--output', '/content/drive/MyDrive/nids-fair-retrain/calibration/infiltration_v1_w30_tcpflags_episode_split_v1/calibration_manifest.json'], returncode=0)

In [6]:
import hashlib
import json

serialized = CALIBRATION_MANIFEST.read_bytes()
manifest = json.loads(serialized)
print('calibration_manifest_sha256:', hashlib.sha256(serialized).hexdigest())
print(json.dumps({
    'counts': manifest['counts'],
    'profile_alignment': manifest['profile_alignment'],
    'candidates': manifest['candidates'],
    'code_revision': manifest['code_revision'],
}, indent=2, sort_keys=True))

calibration_manifest_sha256: fc1c301ae3273a0328dfae41566d4fdbf6b8332e7780ce1b519137ae693f33b1
{
  "candidates": [
    {
      "anchors": [
        "1"
      ],
      "candidate_id": "calibration_01",
      "output_bias_init": -3.3596218658558064,
      "pos_weight": 1.0
    },
    {
      "anchors": [
        "2"
      ],
      "candidate_id": "calibration_02",
      "output_bias_init": -2.666474685295861,
      "pos_weight": 2.0
    },
    {
      "anchors": [
        "sqrt(R)"
      ],
      "candidate_id": "calibration_03",
      "output_bias_init": -1.6798109329279032,
      "pos_weight": 5.364541617057841
    },
    {
      "anchors": [
        "R/2"
      ],
      "candidate_id": "calibration_04",
      "output_bias_init": -0.6931471805599453,
      "pos_weight": 14.389153380572779
    },
    {
      "anchors": [
        "R"
      ],
      "candidate_id": "calibration_05",
      "output_bias_init": 0.0,
      "pos_weight": 28.778306761145558
    }
  ],
  "code_revision": "bf8f1e6

## Checks

In [7]:
result = subprocess.run(
    command,
    cwd=REPO_ROOT,
    check=True,
    capture_output=True,
    text=True,
)
print(result.stdout)

{
  "calibration_manifest_sha256": "fc1c301ae3273a0328dfae41566d4fdbf6b8332e7780ce1b519137ae693f33b1",
  "candidates": [
    {
      "anchors": [
        "1"
      ],
      "candidate_id": "calibration_01",
      "output_bias_init": -3.3596218658558064,
      "pos_weight": 1.0
    },
    {
      "anchors": [
        "2"
      ],
      "candidate_id": "calibration_02",
      "output_bias_init": -2.666474685295861,
      "pos_weight": 2.0
    },
    {
      "anchors": [
        "sqrt(R)"
      ],
      "candidate_id": "calibration_03",
      "output_bias_init": -1.6798109329279032,
      "pos_weight": 5.364541617057841
    },
    {
      "anchors": [
        "R/2"
      ],
      "candidate_id": "calibration_04",
      "output_bias_init": -0.6931471805599453,
      "pos_weight": 14.389153380572779
    },
    {
      "anchors": [
        "R"
      ],
      "candidate_id": "calibration_05",
      "output_bias_init": 0.0,
      "pos_weight": 28.778306761145558
    }
  ],
  "counts": {
    "c

In [8]:
serialized = CALIBRATION_MANIFEST.read_bytes()
manifest = json.loads(serialized)
print('calibration_manifest_sha256:', hashlib.sha256(serialized).hexdigest())
print(json.dumps({
    'counts': manifest['counts'],
    'profile_alignment': manifest['profile_alignment'],
    'candidates': manifest['candidates'],
    'code_revision': manifest['code_revision'],
}, indent=2, sort_keys=True))

calibration_manifest_sha256: fc1c301ae3273a0328dfae41566d4fdbf6b8332e7780ce1b519137ae693f33b1
{
  "candidates": [
    {
      "anchors": [
        "1"
      ],
      "candidate_id": "calibration_01",
      "output_bias_init": -3.3596218658558064,
      "pos_weight": 1.0
    },
    {
      "anchors": [
        "2"
      ],
      "candidate_id": "calibration_02",
      "output_bias_init": -2.666474685295861,
      "pos_weight": 2.0
    },
    {
      "anchors": [
        "sqrt(R)"
      ],
      "candidate_id": "calibration_03",
      "output_bias_init": -1.6798109329279032,
      "pos_weight": 5.364541617057841
    },
    {
      "anchors": [
        "R/2"
      ],
      "candidate_id": "calibration_04",
      "output_bias_init": -0.6931471805599453,
      "pos_weight": 14.389153380572779
    },
    {
      "anchors": [
        "R"
      ],
      "candidate_id": "calibration_05",
      "output_bias_init": 0.0,
      "pos_weight": 28.778306761145558
    }
  ],
  "code_revision": "bf8f1e6

In [9]:
!cd /content/temporalgnn-nids/code/python/tests && \
    PYTHONPATH=.. python -m unittest discover -s . -v

test_empty_windows_are_not_emitted (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_empty_windows_are_not_emitted) ... ok
test_rejects_flow_start_regression_between_chunks (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_rejects_flow_start_regression_between_chunks) ... ok
test_varied_durations_remain_ordered_across_chunk_boundaries (test_build_nfv3_graphs.CompleteWindowIteratorTests.test_varied_durations_remain_ordered_across_chunk_boundaries) ... ok
test_aligned_profiles_read_provenance_only_once (test_build_nfv3_graphs.OutputAuditTests.test_aligned_profiles_read_provenance_only_once) ... ok
test_collection_digest_is_order_independent_and_path_sensitive (test_build_nfv3_graphs.OutputAuditTests.test_collection_digest_is_order_independent_and_path_sensitive) ... ok
test_full_audit_requires_both_binary_classes (test_build_nfv3_graphs.OutputAuditTests.test_full_audit_requires_both_binary_classes) ... ok
test_checkpoint_publishes_mapping_before_state (test_build_nfv3_graph